# Readout quickstart

This notebook builds one small readout example from public TensorDSLab pieces. The numbers are illustrative. Application packages usually provide their own axes, kernels, and workflow wrappers.

## 1. Imports

We begin with Torch, Matplotlib, and the public TensorCore and TensorDSLab objects used below.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch

from tensor_core import (
    CountCoordinates,
    LabelCoordinates,
    NonnegativeInteger,
    OffsetAxis,
    OffsetCoordinates,
    RegularCoordinates,
    Threefry4x32,
)

from tensor_dslab import (
    AnalogGain,
    AnalogGainSpec,
    AnalogWaveform,
    AnalogWaveformConfig,
    AnalogWaveformKernels,
    AnalogWaveformSpec,
    BitDepth,
    BitDepthSpec,
    ChannelAxis,
    Charge,
    ChargeConfig,
    ChargeKernels,
    ChargeSpec,
    DigitizedWaveform,
    DigitizedWaveformConfig,
    DigitizedWaveformKernels,
    DigitizedWaveformSpec,
    EncodedWaveform,
    EncodedWaveformConfig,
    EncodedWaveformKernels,
    EncodedWaveformSpec,
    ExampleAxis,
    FrequencyAxis,
    InputMaximum,
    InputMaximumSpec,
    InputMinimum,
    InputMinimumSpec,
    NoiseWaveform,
    NoiseWaveformConfig,
    NoiseWaveformKernels,
    NoiseWaveformSpec,
    Photoelectrons,
    PhotoelectronsSpec,
    PostTriggerSamples,
    PostTriggerSamplesSpec,
    PowerSpectralDensity,
    PowerSpectralDensitySpec,
    PreTriggerSamples,
    PreTriggerSamplesSpec,
    PulseResponse,
    PulseResponseSpec,
    PureWaveform,
    PureWaveformConfig,
    PureWaveformKernels,
    PureWaveformSpec,
    ReleaseThresholdCode,
    ReleaseThresholdCodeSpec,
    RequiredTimeOverSamples,
    RequiredTimeOverSamplesSpec,
    TimeAxis,
    TriggerThresholdCode,
    TriggerThresholdCodeSpec,
    unit_registry,
)

## 2. Axes

An Example axis permits batching, the Channel axis names three sensors, and the Time axis gives the last tensor dimension a physical spacing. We deliberately use three channels so it is clear that one Product call processes several sensors together in one tensor. Every Product below uses this same ordered domain. The Frequency axis describes the PSD bins used during preparation; it is not a Product dimension.

In [ ]:
device = torch.device("cpu")
field_dtype = torch.float32

example_axis = ExampleAxis(
    coordinates=CountCoordinates(count=1),
)

channel_axis = ChannelAxis(
    coordinates=LabelCoordinates(
        labels=("sensor-0", "sensor-1", "sensor-2"),
    ),
)

time_axis = TimeAxis(
    coordinates=RegularCoordinates(
        start=0,
        step=1,
        count=5000,
    ),
    coordinate_scale=2.0,
    unit=unit_registry.Unit("ns"),
)

frequency_axis = FrequencyAxis(
    coordinates=RegularCoordinates(
        start=0,
        step=1,
        count=2501,
    ),
    coordinate_scale=0.1,
    unit=unit_registry.Unit("MHz"),
)

axes = (
    example_axis,
    channel_axis,
    time_axis,
)

## 3. Photoelectrons

Photoelectrons is the already-produced source for this example. We place four familiar, well-separated deposits across the three sensors so every channel is active and each full pulse remains inside the 10,000 ns window. A real application may obtain this Product from simulation, measurement, or another transformation.

In [ ]:
photoelectrons_spec = PhotoelectronsSpec(
    axes=axes,
    device=device,
    dtype=torch.int64,
    unit=unit_registry.Unit("avalanche"),
)

photoelectron_values = torch.zeros(
    photoelectrons_spec.shape,
    dtype=torch.int64,
    device=device,
)
photoelectron_values[0, 0, 100] = 1
photoelectron_values[0, 0, 3700] = 4
photoelectron_values[0, 1, 1300] = 2
photoelectron_values[0, 2, 2500] = 3
photoelectron_values_before = photoelectron_values.clone()

photoelectrons = Photoelectrons(
    tensor=photoelectron_values,
    spec=photoelectrons_spec,
)

## 4. Charge

An empty ChargeKernels collection leaves these source counts unchanged apart from the requested floating representation. This is a deliberate minimal starting point. Users may later add physical Charge mechanisms with public Kernels.

In [ ]:
charge_spec = ChargeSpec(
    axes=axes,
    device=device,
    dtype=field_dtype,
    unit=unit_registry.Unit("avalanche"),
)

charge_config = ChargeConfig(
    spec=charge_spec,
    kernels=ChargeKernels(members=()),
    correlated_avalanche_generations=NonnegativeInteger(value=0),
)

rng = Threefry4x32(seed=2026)

charge = Charge.create(
    sources=(photoelectrons,),
    config=charge_config,
    rng=rng,
)

## 5. Pure waveform

PulseResponse maps avalanches into voltage samples. Its empty conditioning geometry shares one response across all three sensors. The transparent Gaussian-and-double-error-function template is recognizable from earlier TensorDSLab examples, but these numerical values remain illustrative rather than calibrated. The deposits leave the complete 1,011-sample response inside the window.

In [ ]:
pulse_support_ns = 2020.27
pulse_coefficient_count = math.ceil(
    pulse_support_ns / time_axis.coordinate_scale
)
pulse_offsets = torch.arange(
    pulse_coefficient_count,
    dtype=field_dtype,
    device=device,
)
pulse_time_ns = pulse_offsets * time_axis.coordinate_scale
pulse_x = pulse_time_ns - 232.89

pulse_gaussian = torch.exp(
    -(pulse_x**2) / (2.0 * 507.72**2)
) / math.sqrt(2.0 * math.pi * 507.72**2)
pulse_first = 1.0 + torch.erf(
    (pulse_x - (-81.92)) / (math.sqrt(2.0) * 147.28)
)
pulse_second = 1.0 + torch.erf(
    (pulse_x - (-176.50)) / (math.sqrt(2.0) * 45.69)
)

pulse_raw = pulse_gaussian * pulse_first * pulse_second
pulse_values = (
    pulse_raw / torch.max(torch.abs(pulse_raw)) * -14.5912372
)

pulse_time_axis = OffsetAxis(
    coordinates=OffsetCoordinates(offsets=tuple(range(1011))),
    relative_to=TimeAxis,
)

pulse_response = PulseResponse(
    tensor=pulse_values,
    spec=PulseResponseSpec(
        conditioning_axes=(),
        operation_axes=(pulse_time_axis,),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV / avalanche"),
    ),
)

pure_waveform_config = PureWaveformConfig(
    spec=PureWaveformSpec(
        axes=axes,
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
    kernels=PureWaveformKernels(
        members=(pulse_response,),
    ),
)

pure_waveform = PureWaveform.create(
    sources=(charge,),
    config=pure_waveform_config,
)

## 6. Noise waveform

NoiseWaveform has no source Product here. Instead, three literal PSD rows give each sensor a different electronic-noise shape. The Channel axis conditions the rows, while the Frequency axis describes the one-sided bins. Preparation checks that this frequency grid matches the output Time axis before drawing noise.

In [ ]:
psd_sensor_0 = torch.cat(
    (
        torch.zeros(1, dtype=field_dtype, device=device),
        torch.full((625,), 0.012, dtype=field_dtype, device=device),
        torch.full((1875,), 0.004, dtype=field_dtype, device=device),
    )
)
psd_sensor_1 = torch.cat(
    (
        torch.zeros(1, dtype=field_dtype, device=device),
        torch.full((938,), 0.008, dtype=field_dtype, device=device),
        torch.full((1562,), 0.016, dtype=field_dtype, device=device),
    )
)
psd_sensor_2 = torch.cat(
    (
        torch.zeros(1, dtype=field_dtype, device=device),
        torch.full((1250,), 0.020, dtype=field_dtype, device=device),
        torch.full((1250,), 0.006, dtype=field_dtype, device=device),
    )
)

psd_values = torch.stack(
    (
        psd_sensor_0,
        psd_sensor_1,
        psd_sensor_2,
    )
)

psd_spec = PowerSpectralDensitySpec(
    conditioning_axes=(channel_axis,),
    operation_axes=(frequency_axis,),
    device=device,
    dtype=field_dtype,
    unit=unit_registry.Unit("mV ** 2"),
)

power_spectral_density = PowerSpectralDensity(
    tensor=psd_values,
    spec=psd_spec,
)

noise_waveform_config = NoiseWaveformConfig(
    spec=NoiseWaveformSpec(
        axes=axes,
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
    kernels=NoiseWaveformKernels(
        members=(power_spectral_density,),
    ),
)

noise_waveform = NoiseWaveform.create(
    sources=(),
    config=noise_waveform_config,
    rng=rng,
)

## 7. Analog waveform

AnalogWaveform combines the pure response and the electronic noise. Empty saturation Kernels keep this example focused on ordinary source composition.

In [ ]:
analog_waveform_config = AnalogWaveformConfig(
    spec=AnalogWaveformSpec(
        axes=axes,
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
    kernels=AnalogWaveformKernels(members=()),
)

analog_waveform = AnalogWaveform.create(
    sources=(
        pure_waveform,
        noise_waveform,
    ),
    config=analog_waveform_config,
)

## 8. Digitized waveform

Four scalar Kernels describe an illustrative 12-bit digitizer with a -80 mV to 20 mV input interval and unit linear gain. The wider interval keeps the familiar pulse visible without rail clipping; it is a demo choice, not calibration. The Kernels apply globally here, while an application may condition the same public coefficient types on Channel or Example axes when its hardware varies.

In [ ]:
bit_depth = BitDepth(
    tensor=torch.tensor(12, dtype=torch.int16, device=device),
    spec=BitDepthSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int16,
    ),
)

input_minimum = InputMinimum(
    tensor=torch.tensor(-80.0, dtype=field_dtype, device=device),
    spec=InputMinimumSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
)

input_maximum = InputMaximum(
    tensor=torch.tensor(20.0, dtype=field_dtype, device=device),
    spec=InputMaximumSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit("mV"),
    ),
)

analog_gain = AnalogGain(
    tensor=torch.tensor(1.0, dtype=field_dtype, device=device),
    spec=AnalogGainSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=field_dtype,
        unit=unit_registry.Unit(""),
    ),
)

digitized_waveform_config = DigitizedWaveformConfig(
    spec=DigitizedWaveformSpec(
        axes=axes,
        device=device,
        dtype=torch.int32,
        unit=unit_registry.Unit(""),
    ),
    kernels=DigitizedWaveformKernels(
        members=(
            bit_depth,
            input_minimum,
            input_maximum,
            analog_gain,
        ),
    ),
)

digitized_waveform = DigitizedWaveform.create(
    sources=(analog_waveform,),
    config=digitized_waveform_config,
)

## 9. Encoded waveform

EncodedWaveform shows which ADC samples a DAQ-like zero-length-encoding stage keeps. Retained values remain exact ADC codes; the configured negative value means that no code was retained at that sample. The five policy Kernels are global in this small example, although applications may condition them on non-Time roles. These settings are illustrative rather than detector calibration.

In [ ]:
trigger_threshold_code = TriggerThresholdCode(
    tensor=torch.tensor(2500, dtype=torch.int64, device=device),
    spec=TriggerThresholdCodeSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
release_threshold_code = ReleaseThresholdCode(
    tensor=torch.tensor(2800, dtype=torch.int64, device=device),
    spec=ReleaseThresholdCodeSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
required_time_over_samples = RequiredTimeOverSamples(
    tensor=torch.tensor(3, dtype=torch.int64, device=device),
    spec=RequiredTimeOverSamplesSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
pre_trigger_samples = PreTriggerSamples(
    tensor=torch.tensor(25, dtype=torch.int64, device=device),
    spec=PreTriggerSamplesSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)
post_trigger_samples = PostTriggerSamples(
    tensor=torch.tensor(50, dtype=torch.int64, device=device),
    spec=PostTriggerSamplesSpec(
        conditioning_axes=(),
        operation_axes=(),
        device=device,
        dtype=torch.int64,
    ),
)

encoded_waveform_config = EncodedWaveformConfig(
    spec=EncodedWaveformSpec(
        axes=axes,
        device=device,
        dtype=torch.int32,
        unit=unit_registry.Unit(""),
        suppression_code=-1,
    ),
    kernels=EncodedWaveformKernels(
        members=(
            trigger_threshold_code,
            release_threshold_code,
            required_time_over_samples,
            pre_trigger_samples,
            post_trigger_samples,
        ),
    ),
)

encoded_waveform = EncodedWaveform.create(
    sources=(digitized_waveform,),
    config=encoded_waveform_config,
)

## 10. Shared shape

Each Product represents a different quantity or transformation, but all seven use the same `(example, channel, time)` domain. These light checks make that shared shape visible without repeating Product validation.

In [ ]:
expected_shape = tuple(axis.size for axis in axes)

assert photoelectrons.tensor.shape == expected_shape
assert charge.tensor.shape == expected_shape
assert pure_waveform.tensor.shape == expected_shape
assert noise_waveform.tensor.shape == expected_shape
assert analog_waveform.tensor.shape == expected_shape
assert digitized_waveform.tensor.shape == expected_shape
assert encoded_waveform.tensor.shape == expected_shape

## 11. Product views

The first two panels match because no Charge mechanisms were enabled. The pulse panel shows the deterministic sensor response, the noise panel shows the PSD-driven electronic component, the analog panel combines those waveforms, and the digitized panel shows integer ADC codes. The last panel leaves suppressed regions blank. Every visible segment is an unchanged ADC-code interval retained by the ZLE stage. Plotting libraries receive ordinary CPU values at this presentation boundary.

Applications may keep these separate Product values, plot them, or pass them into another Product as their workflow requires.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

channel_labels = channel_axis.coordinates.labels
sensor_colors = ("tab:blue", "tab:orange", "tab:green")
time_ns = [
    float(time_axis.quantity_at(index).to("ns").magnitude)
    for index in range(time_axis.size)
]

products = (
    photoelectrons,
    charge,
    pure_waveform,
    noise_waveform,
    analog_waveform,
    digitized_waveform,
    encoded_waveform,
)
y_labels = (
    "Photoelectrons",
    "Charge (avalanche)",
    "Pure (mV)",
    "Noise (mV)",
    "Analog (mV)",
    "ADC code",
    "Retained ADC code",
)
step_panels = {0, 1, 5, 6}

figure, plot_axes = plt.subplots(
    7,
    1,
    figsize=(11, 15),
    sharex=True,
    constrained_layout=True,
)

for panel_index, (plot_axis, product, y_label) in enumerate(
    zip(plot_axes, products, y_labels)
):
    for channel_index, (channel_label, color) in enumerate(
        zip(channel_labels, sensor_colors)
    ):
        values = product.tensor[0, channel_index].detach().cpu().tolist()
        if panel_index == 6:
            values = [
                float("nan")
                if value == encoded_waveform.spec.suppression_code
                else value
                for value in values
            ]
        if panel_index in step_panels:
            plot_axis.step(
                time_ns,
                values,
                where="post",
                color=color,
                label=channel_label,
            )
        else:
            plot_axis.plot(
                time_ns,
                values,
                color=color,
                label=channel_label,
            )
    plot_axis.set_ylabel(y_label)

plot_axes[-1].set_xlabel("Time (ns)")
figure.suptitle(
    "Illustrative three-sensor readout products",
    x=0.01,
    ha="left",
)
figure.legend(
    plot_axes[0].lines,
    channel_labels,
    loc="upper right",
    ncol=3,
    frameon=False,
)
plt.show()